# Grammar score from speech

This notebook gives a grammar score to a spoken English answer. The score is a number from 1.0 to 5.0. The metric is the mean absolute error (MAE).

## Flow

1. **Transcripts.** Whisper changes each audio file into text. A prompt keeps the fillers, the repeats and the false starts. The notebook also keeps the pause data and the confidence data of Whisper.
2. **Features.** The notebook calculates word counts, sentence lengths, filler rates, speech rate and pause ratios. Only the models in step 4 read these features. The encoders read the transcript.
3. **Encoders.** Two transformer encoders learn the score from the transcript. Each encoder trains one time for each cross-validation fold. Each fold gives predictions for its validation rows and for the test rows.
4. **Combination.** Seven candidates compete: the two single encoders, a mean, a weighted blend, a ridge model and two gradient-boosted models. Repeated cross-validation selects the candidate with the lowest MAE.
5. **Submission.** The notebook limits each prediction to the label range. It then moves each prediction to the nearest label on the 0.5 grid and writes the submission file.

The notebook writes all transcripts and all fold predictions to `RUN_DIR`. An interrupted session thus continues from the last fold. A change to a training setting makes the cached predictions incorrect. Remove the model folder in `RUN_DIR` before you start the notebook again.

Each fold selects its best epoch on its own validation rows. The out-of-fold MAE is thus a little better than the true value. Use the same procedure for each experiment, and let the leaderboard give the final answer.


## 0. Environment

The notebook uses the pinned `transformers` version. The encoder settings below apply to that version.


In [1]:
!pip install -q openai-whisper tqdm transformers==4.40.0 scipy scikit-learn sentencepiece protobuf soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


## 1. Settings and paths

This cell holds every setting of the run. `RUN_DIR` holds the transcript cache, the fold predictions and all the submission files.


In [2]:
import gc
import json
import math
import random
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import whisper
from scipy.optimize import minimize
from scipy.stats import pearsonr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import RepeatedKFold, StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

warnings.filterwarnings('ignore', category=FutureWarning)

# ----- Run settings -----
KEEP_ZERO_LABELS = False # False removes the rows with label 0 and the rows with an empty transcript
SEED = 42
N_FOLDS = 4              # number of cross-validation folds
N_SEEDS = 1              # number of random seeds for each fold
EPOCHS = 6               # the cosine schedule covers exactly this number of epochs
PATIENCE = 3             # the fold stops after this number of epochs without a lower validation MAE
WARMUP_FRAC = 0.1
MAX_TOKEN_LEN = 512      # token limit; each batch is padded to its longest transcript only
WEIGHT_DECAY = 0.01
LOSS_BETA = 0.3          # SmoothL1 beta; a smaller value is nearer to pure L1 loss
DISABLE_DROPOUT = True   # True sets all dropout probabilities in the encoder to zero
USE_AMP = True           # True uses fp16 autocast on a GPU

WHISPER_MODEL = 'small'
USE_DISFLUENCY_PROMPT = True   # True gives Whisper a prompt that keeps the fillers and the repeats
DISFLUENCY_PROMPT = "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking. I I was going to, uh, to say, yeah."

RUN_MODELS = ['deberta_v3_large', 'roberta_large']
MODEL_CONFIGS = {
    'deberta_v3_large': dict(checkpoint='microsoft/deberta-v3-large', batch_size=4, accum=4, lr=2e-5, llrd=0.95, head_lr=2e-4),
    'roberta_large':    dict(checkpoint='roberta-large',              batch_size=8, accum=2, lr=1.5e-5, llrd=0.95, head_lr=2e-4),
}

# ----- Paths -----
DATASET_NAME = 'shl-hiring-assessment-2026'
BASE_DIR = Path(f'/kaggle/input/competitions/{DATASET_NAME}/Dataset_Final')
TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV = BASE_DIR / 'test.csv'
TRAIN_AUDIO = BASE_DIR / 'train'
TEST_AUDIO = BASE_DIR / 'test'

RUN_DIR = Path('/kaggle/working/model_comparison_v2_trim')
TAG = 'disfl' if USE_DISFLUENCY_PROMPT else 'plain'
TRANSCRIPT_DIR = RUN_DIR / 'transcripts'
RUN_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_TRANSCRIPTS = TRANSCRIPT_DIR / f'train_transcripts_{TAG}.csv'
TEST_TRANSCRIPTS = TRANSCRIPT_DIR / f'test_transcripts_{TAG}.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ON = USE_AMP and DEVICE.type == 'cuda'


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print(f'Device: {DEVICE} | AMP: {AMP_ON} | keep label==0: {KEEP_ZERO_LABELS} | folds: {N_FOLDS} | epochs/fold: {EPOCHS} | models: {RUN_MODELS}')

Device: cuda | AMP: True | keep label==0: False | folds: 4 | epochs/fold: 6 | models: ['deberta_v3_large', 'roberta_large']


## 2. Transcripts and speech features

Whisper removes the fillers, the stutters and the false starts by default. These items are evidence of the grammar quality. The initial prompt therefore tells Whisper to keep them.

The cell also keeps the confidence of each segment, the length of each pause and the duration of the audio. The models in section 6 use these values.


In [3]:
WHISPER_COLS = ['seg_logprob_mean', 'seg_logprob_min', 'seg_no_speech_max', 'seg_compression_mean',
                'n_segments', 'pause_total', 'pause_max', 'n_long_pauses', 'speech_span']
CACHE_COLS = ['filename', 'transcript', 'duration'] + WHISPER_COLS

_whisper_model = None


def get_whisper():
    """Return the Whisper model. Load it on the first call only."""
    global _whisper_model
    if _whisper_model is None:
        _whisper_model = whisper.load_model(WHISPER_MODEL)
    return _whisper_model


def audio_duration(path):
    """Return the length of an audio file in seconds."""
    try:
        import soundfile as sf
        return float(sf.info(str(path)).duration)
    except Exception:
        return len(whisper.load_audio(str(path))) / 16000.0


def segment_features(segments):
    """Calculate the confidence statistics and the pause statistics of Whisper segments."""
    if not segments:
        return {c: np.nan for c in WHISPER_COLS}
    seg_lengths = np.array([max(s['end'] - s['start'], 1e-3) for s in segments])
    log_probs = np.array([s['avg_logprob'] for s in segments])
    pauses = np.array([max(segments[i + 1]['start'] - segments[i]['end'], 0.0) for i in range(len(segments) - 1)]
                    if len(segments) > 1 else [0.0])
    return {
        'seg_logprob_mean': float(np.average(log_probs, weights=seg_lengths)),
        'seg_logprob_min': float(log_probs.min()),
        'seg_no_speech_max': float(max(s['no_speech_prob'] for s in segments)),
        'seg_compression_mean': float(np.mean([s['compression_ratio'] for s in segments])),
        'n_segments': len(segments),
        'pause_total': float(pauses.sum()),
        'pause_max': float(pauses.max()),
        'n_long_pauses': int((pauses > 0.5).sum()),
        'speech_span': float(segments[-1]['end'] - segments[0]['start']),
    }


def transcribe_files(csv_path, audio_dir, output_path):
    """Transcribe each audio file and write the results to a cache file.

    The function reads the cache first and transcribes only the files that are not in it.
    It writes the cache again after each 10 files.
    """
    rows = pd.read_csv(csv_path)
    records = {}
    if output_path.exists():
        previous = pd.read_csv(output_path)
        if all(c in previous.columns for c in CACHE_COLS):
            previous['transcript'] = previous['transcript'].fillna('').astype(str)
            previous = previous[previous['transcript'].str.strip() != '']
            records = previous.set_index('filename')[CACHE_COLS[1:]].to_dict('index')

    pending = rows[~rows['filename'].isin(records)].reset_index(drop=True)
    print(f'{output_path.name}: {len(records)} cached, {len(pending)} to transcribe')

    def save(final=False):
        """Write the cache. With final=True, keep a row for each file in the index."""
        df = pd.DataFrame([{'filename': k, **v} for k, v in records.items()], columns=CACHE_COLS)
        if final:
            df = rows.merge(df, on='filename', how='left')
        df.to_csv(output_path, index=False)

    for i, row in enumerate(tqdm(pending.itertuples(index=False), total=len(pending), desc=f'Transcribing {output_path.stem}'), start=1):
        path = audio_dir / row.filename
        try:
            kwargs = dict(language='en', fp16=(DEVICE.type == 'cuda'), condition_on_previous_text=False, temperature=0)
            if USE_DISFLUENCY_PROMPT:
                kwargs['initial_prompt'] = DISFLUENCY_PROMPT
            result = get_whisper().transcribe(str(path), **kwargs)
            records[row.filename] = {'transcript': result['text'].strip(), 'duration': audio_duration(path),
                                     **segment_features(result.get('segments', []))}
        except Exception as error:
            print(f'Could not transcribe {row.filename}: {error}')
            records[row.filename] = {'transcript': '', 'duration': np.nan, **{c: np.nan for c in WHISPER_COLS}}
        if i % 10 == 0:
            save()
    save(final=True)
    return pd.read_csv(output_path)


train_transcripts = transcribe_files(TRAIN_CSV, TRAIN_AUDIO, TRAIN_TRANSCRIPTS)
test_transcripts = transcribe_files(TEST_CSV, TEST_AUDIO, TEST_TRANSCRIPTS)

global_whisper = _whisper_model
_whisper_model = None
del global_whisper
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()

for df in (train_transcripts, test_transcripts):
    df['transcript'] = df['transcript'].fillna('').astype(str)
print(f'Train rows: {len(train_transcripts)} (empty: {(train_transcripts.transcript.str.strip() == "").sum()}) | '
      f'Test rows: {len(test_transcripts)} (empty: {(test_transcripts.transcript.str.strip() == "").sum()})')

train_transcripts_disfl.csv: 0 cached, 769 to transcribe


Transcribing train_transcripts_disfl:   0%|          | 0/769 [00:00<?, ?it/s]


  0%|                                               | 0.00/461M [00:00<?, ?iB/s]
  0%|                                      | 1.00M/461M [00:00<00:46, 10.4MiB/s]
  3%|█▎                                    | 16.0M/461M [00:00<00:04, 96.4MiB/s]
  6%|██▍                                    | 29.0M/461M [00:00<00:03, 115MiB/s]
  9%|███▍                                   | 39.9M/461M [00:00<00:03, 113MiB/s]
 11%|████▎                                  | 50.8M/461M [00:00<00:03, 110MiB/s]
 14%|█████▍                                 | 64.1M/461M [00:00<00:03, 120MiB/s]
 16%|██████▍                                | 75.8M/461M [00:00<00:03, 120MiB/s]
 19%|███████▍                               | 88.5M/461M [00:00<00:03, 124MiB/s]
 22%|████████▋                               | 100M/461M [00:00<00:03, 116MiB/s]
 24%|█████████▋                              | 112M/461M [00:01<00:03, 117MiB/s]
 27%|██████████▋                             | 123M/461M [00:01<00:03, 115MiB/s]
 29%|███████████▋          

test_transcripts_disfl.csv: 0 cached, 216 to transcribe


Transcribing test_transcripts_disfl:   0%|          | 0/216 [00:00<?, ?it/s]

Train rows: 769 (empty: 0) | Test rows: 216 (empty: 0)


## 3. Text cleanup, features and folds

The cell removes the extra whitespace from each transcript. It then removes the rows with no label. With `KEEP_ZERO_LABELS = False`, it also removes the rows with label 0 and the rows with an empty transcript.

The folds keep the same label distribution. Each fold thus contains the full range of scores.


In [4]:
FILLERS = re.compile(r"\b(?:um+|uh+|er+|erm|ah+|hmm+|mm+)\b", re.I)
REPEATS = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)


def clean_text(text):
    """Replace each run of whitespace with one space."""
    return re.sub(r'\s+', ' ', str(text or '')).strip()


def text_features(text):
    """Calculate the length statistics, the variety statistics and the disfluency counts of a transcript."""
    words = re.findall(r"[A-Za-z']+", text)
    lower_words = [w.lower() for w in words]
    n = len(words)
    sentences = [s for s in re.split(r'[.!?]+', text) if s.strip()]
    return {
        'n_words': n,
        'n_sents': len(sentences),
        'avg_sent_len': n / max(len(sentences), 1),
        'avg_word_len': float(np.mean([len(w) for w in words])) if n else 0.0,
        'ttr': len(set(lower_words)) / n if n else 0.0,
        'n_fillers': len(FILLERS.findall(text)),
        'n_repeats': len(REPEATS.findall(text)),
        'n_like': lower_words.count('like'),
    }


def build_features(df):
    """Join the text features, the Whisper features and the rate features into one table."""
    text_frame = pd.DataFrame([text_features(t) for t in df['transcript']])
    out = pd.concat([text_frame, df[['duration'] + WHISPER_COLS].reset_index(drop=True)], axis=1)
    out['wpm'] = out['n_words'] / out['duration'].clip(lower=1.0) * 60
    out['filler_rate'] = out['n_fillers'] / out['n_words'].clip(lower=1)
    out['repeat_rate'] = out['n_repeats'] / out['n_words'].clip(lower=1)
    out['pause_ratio'] = out['pause_total'] / out['speech_span'].clip(lower=1.0)
    return out


train_rows = train_transcripts.copy()
train_rows['label'] = pd.to_numeric(train_rows['label'], errors='coerce')
train_rows['transcript'] = train_rows['transcript'].map(clean_text)
test_rows = test_transcripts.copy()
test_rows['transcript'] = test_rows['transcript'].map(clean_text)

if KEEP_ZERO_LABELS:
    keep = train_rows['label'].notna()
else:
    keep = train_rows['transcript'].ne('') & train_rows['label'].notna() & (train_rows['label'] > 0)
train_rows = train_rows[keep].reset_index(drop=True)
n_zero = int((train_rows['label'] == 0).sum())
n_empty = int(train_rows['transcript'].eq('').sum())
n_both = int(((train_rows['label'] == 0) & train_rows['transcript'].eq('')).sum())
print(f'Rows with label 0: {n_zero} | rows with empty transcript: {n_empty} | both: {n_both}')

feat_train = build_features(train_rows)
feat_test = build_features(test_rows)
FEATURE_COLS = list(feat_train.columns)

y = train_rows['label'].values.astype(float)
LABEL_LO, LABEL_HI = float(y.min()), float(y.max())
print(f'Training rows: {len(train_rows)} | label range [{LABEL_LO}, {LABEL_HI}] | mean {y.mean():.3f} | std {y.std():.3f}')
print('Label distribution:\n', train_rows['label'].round(1).value_counts().sort_index().to_string())


def make_bins(labels, n_splits):
    """Put the labels into integer bins. Merge a bin into its nearest neighbour if it holds fewer rows than n_splits."""
    bins = np.clip(np.floor(labels), 0, 5).astype(int)
    counts = pd.Series(bins).value_counts().to_dict()
    while min(counts.values()) < n_splits and len(counts) > 1:
        smallest = min(counts, key=counts.get)
        nearest = min((b for b in counts if b != smallest), key=lambda b: abs(b - smallest))
        bins[bins == smallest] = nearest
        counts = pd.Series(bins).value_counts().to_dict()
    return bins


bins = make_bins(y, N_FOLDS)
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(train_rows, bins))
print('Fold sizes (train/val):', [(len(a), len(b)) for a, b in folds])

Rows with label 0: 0 | rows with empty transcript: 0 | both: 0
Training rows: 732 | label range [1.0, 5.0] | mean 3.481 | std 1.014
Label distribution:
 label
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Fold sizes (train/val): [(549, 183), (549, 183), (549, 183), (549, 183)]


## 4. Model and training loop

The model is a transformer encoder, a masked mean pooling layer and a linear head.

The training loop uses a SmoothL1 loss on the standardised labels. It uses a layer-wise learning-rate decay, a cosine schedule with a warmup, and mixed precision. The loop measures the validation MAE after each epoch. It keeps the weights of the best epoch and stops early after `PATIENCE` epochs with no lower MAE.


In [5]:
class TextDataset(Dataset):
    """Tokenised transcripts and their targets. Tokenisation happens once, in the constructor."""

    def __init__(self, texts, targets, tokenizer):
        self.enc = tokenizer(list(texts), max_length=MAX_TOKEN_LEN, truncation=True)
        self.targets = [float(t) for t in targets]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        return {'input_ids': self.enc['input_ids'][i],
                'attention_mask': self.enc['attention_mask'][i],
                'target': self.targets[i]}


def make_collate(tokenizer):
    """Return a collate function. The function pads each batch to its longest sequence."""

    def collate(batch):
        padded = tokenizer.pad({'input_ids': [b['input_ids'] for b in batch],
                                'attention_mask': [b['attention_mask'] for b in batch]}, return_tensors='pt')
        return {'input_ids': padded['input_ids'],
                'attention_mask': padded['attention_mask'],
                'target': torch.tensor([b['target'] for b in batch], dtype=torch.float32)}
    return collate


class Regressor(nn.Module):
    """A transformer encoder, a masked mean pooling layer and a linear head."""

    def __init__(self, checkpoint):
        super().__init__()
        config = AutoConfig.from_pretrained(checkpoint)
        if DISABLE_DROPOUT:
            config.hidden_dropout_prob = 0.0
            config.attention_probs_dropout_prob = 0.0
        self.backbone = AutoModel.from_pretrained(checkpoint, config=config)
        self.head = nn.Linear(config.hidden_size, 1)
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, input_ids, attention_mask):
        hidden = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state.float()
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        return self.head(pooled).squeeze(-1)


def make_param_groups(model, lr, llrd, head_lr, weight_decay):
    """Group the parameters for layer-wise learning-rate decay.

    The top encoder layer gets `lr`. Each layer below it gets `llrd` times the rate of the layer above.
    The head gets `head_lr`. Biases and normalisation weights get no weight decay.
    """
    n_layers = model.backbone.config.num_hidden_layers
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith('head.'):
            group_lr = head_lr
        else:
            if name.startswith('backbone.embeddings'):
                depth = 0
            else:
                match = re.search(r'encoder\.layer\.(\d+)\.', name)
                depth = int(match.group(1)) + 1 if match else n_layers
            group_lr = lr * (llrd ** (n_layers - depth))
        no_decay = name.endswith('.bias') or 'LayerNorm' in name or 'layer_norm' in name
        groups.setdefault((group_lr, 0.0 if no_decay else weight_decay), []).append(param)
    return [{'params': ps, 'lr': l, 'weight_decay': w} for (l, w), ps in groups.items()]


@torch.no_grad()
def predict(model, loader):
    """Return the model outputs for every batch of a loader, in standardised units."""
    model.eval()
    outputs = []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ON):
            out = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
        outputs.append(out.float().cpu().numpy())
    return np.concatenate(outputs)


def score(actual, predicted):
    """Return the MAE, the RMSE and the Pearson correlation of a set of predictions."""
    predicted = np.asarray(predicted, dtype=float)
    pearson = pearsonr(actual, predicted)[0] if np.std(predicted) > 0 else 0.0
    return {'mae': float(mean_absolute_error(actual, predicted)),
            'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
            'pearson': float(pearson)}


def train_fold(cfg, tokenizer, tr_idx, va_idx, seed, tag):
    """Train one encoder on one fold.

    The targets are standardised with the mean and the standard deviation of the training rows.
    The fold keeps the weights of the epoch with the lowest validation MAE.
    It stops early after PATIENCE epochs without a lower MAE.
    Returns the validation predictions, the test predictions, the epoch history and the best epoch.
    """
    set_seed(seed)
    collate = make_collate(tokenizer)
    tr_y, va_y = y[tr_idx], y[va_idx]
    mu, sd = tr_y.mean(), tr_y.std()

    train_ds = TextDataset(train_rows['transcript'].values[tr_idx], (tr_y - mu) / sd, tokenizer)
    valid_ds = TextDataset(train_rows['transcript'].values[va_idx], np.zeros(len(va_idx)), tokenizer)
    test_ds = TextDataset(test_rows['transcript'].values, np.zeros(len(test_rows)), tokenizer)
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, collate_fn=collate)
    valid_loader = DataLoader(valid_ds, batch_size=16, shuffle=False, collate_fn=collate)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate)

    model = Regressor(cfg['checkpoint']).to(DEVICE)
    optimizer = torch.optim.AdamW(
        make_param_groups(model, cfg['lr'], cfg['llrd'], cfg['head_lr'], WEIGHT_DECAY), lr=cfg['lr'], eps=1e-6)
    steps_per_epoch = math.ceil(len(train_loader) / cfg['accum'])
    total_steps = steps_per_epoch * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)
    try:
        scaler = torch.amp.GradScaler('cuda', enabled=AMP_ON)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=AMP_ON)
    loss_fn = nn.SmoothL1Loss(beta=LOSS_BETA)

    best_mae, best_epoch, best_state, bad_epochs = float('inf'), 0, None, 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running = 0.0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{tag} ep{epoch}/{EPOCHS}', leave=False), start=1):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ON):
                prediction = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE))
            loss = loss_fn(prediction.float(), batch['target'].to(DEVICE))
            scaler.scale(loss / cfg['accum']).backward()
            if step % cfg['accum'] == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            running += loss.item()

        val_pred = np.clip(np.nan_to_num(predict(model, valid_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)
        metrics = score(va_y, val_pred)
        history.append({'epoch': epoch, 'train_loss': running / len(train_loader), **metrics})
        print(f'{tag} | ep {epoch:>2} | loss={running / len(train_loader):.4f} | MAE={metrics["mae"]:.4f} | '
              f'RMSE={metrics["rmse"]:.4f} | r={metrics["pearson"]:.4f}')

        if metrics['mae'] < best_mae or best_state is None:
            best_mae, best_epoch, bad_epochs = metrics['mae'], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'{tag} | early stop (no MAE gain for {PATIENCE} epochs)')
                break

    model.load_state_dict(best_state)
    val_pred = np.clip(np.nan_to_num(predict(model, valid_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)
    test_pred = np.clip(np.nan_to_num(predict(model, test_loader) * sd + mu, nan=mu), LABEL_LO, LABEL_HI)

    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return val_pred, test_pred, history, best_epoch


def run_model(name):
    """Train one encoder on every fold and every seed.

    Each fold writes its predictions to RUN_DIR, so an interrupted run continues from the last fold.
    Returns the out-of-fold predictions and the fold-averaged test predictions.
    """
    cfg = MODEL_CONFIGS[name]
    model_dir = RUN_DIR / name
    model_dir.mkdir(parents=True, exist_ok=True)
    final_path = model_dir / 'cv_predictions.npz'
    if final_path.exists():
        data = np.load(final_path)
        print(f'{name}: loaded cached CV predictions')
        return data['oof'], data['test']

    tokenizer = AutoTokenizer.from_pretrained(cfg['checkpoint'])
    oof = np.zeros((N_SEEDS, len(train_rows)))
    test_preds = []
    for s in range(N_SEEDS):
        for fold, (tr_idx, va_idx) in enumerate(folds):
            fold_path = model_dir / f'seed{s}_fold{fold}.npz'
            if fold_path.exists():
                data = np.load(fold_path)
                val_pred, test_pred = data['val'], data['test']
                print(f'{name} seed{s} fold{fold}: cached | MAE={mean_absolute_error(y[va_idx], val_pred):.4f}')
            else:
                tag = f'{name} s{s} f{fold}'
                val_pred, test_pred, history, best_epoch = train_fold(cfg, tokenizer, tr_idx, va_idx, SEED + 1000 * s + fold, tag)
                np.savez(fold_path, val=val_pred, test=test_pred)
                json.dump(history, open(model_dir / f'seed{s}_fold{fold}_history.json', 'w'))
                print(f'==> {tag}: best epoch {best_epoch} | MAE={mean_absolute_error(y[va_idx], val_pred):.4f}')
            oof[s, va_idx] = val_pred
            test_preds.append(test_pred)
    oof_mean, test_mean = oof.mean(0), np.mean(test_preds, axis=0)
    np.savez(final_path, oof=oof_mean, test=test_mean)
    return oof_mean, test_mean

## 5. Cross-validation run

Each encoder trains one time for each fold. The out-of-fold predictions give an MAE for all the training rows. The test predictions of the folds are averaged.


In [6]:
oof_preds, test_preds, summary = {}, {}, []
for name in RUN_MODELS:
    print(f'\n===== {name} ({MODEL_CONFIGS[name]["checkpoint"]}) =====')
    oof_preds[name], test_preds[name] = run_model(name)
    metrics = score(y, oof_preds[name])
    fold_maes = [mean_absolute_error(y[va], oof_preds[name][va]) for _, va in folds]
    summary.append({'model': name, **{f'oof_{k}': v for k, v in metrics.items()}, 'fold_mae_std': float(np.std(fold_maes))})
    pd.DataFrame(summary).to_csv(RUN_DIR / 'model_comparison.csv', index=False)

oof_df = pd.DataFrame(oof_preds)
test_df = pd.DataFrame(test_preds)
display(pd.DataFrame(summary).sort_values('oof_mae'))


===== deberta_v3_large (microsoft/deberta-v3-large) =====


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


...

roberta_large s0 f3 ep5/6:   0%|          | 0/69 [00:00<?, ?it/s]

roberta_large s0 f3 | ep  5 | loss=0.1179 | MAE=0.4326 | RMSE=0.5328 | r=0.8552


roberta_large s0 f3 ep6/6:   0%|          | 0/69 [00:00<?, ?it/s]

roberta_large s0 f3 | ep  6 | loss=0.0856 | MAE=0.4219 | RMSE=0.5236 | r=0.8583
==> roberta_large s0 f3: best epoch 6 | MAE=0.4219


,model,oof_mae,oof_rmse,oof_pearson,fold_mae_std
0,deberta_v3_large,0.452978,0.58088,0.820900,0.027910
1,roberta_large,0.486369,0.61698,0.796897,0.037287


## 6. Combination of the models

The candidates are the single encoders, a mean of the encoders, a weighted blend with the lowest MAE, a ridge model, and two gradient-boosted models that also read the fluency features. Repeated 5-fold cross-validation selects the candidate with the lowest MAE.

The report also contains a model that reads the features only. That model is a baseline, and the selection step ignores it.


In [7]:
class RowMeanModel:
    """Predicts the mean of its input columns."""

    def fit(self, X, y_):
        return self

    def predict(self, X):
        return np.asarray(X).mean(axis=1)


class ColumnSubsetModel:
    """Fits an estimator on a named subset of the columns."""

    def __init__(self, estimator, cols):
        self.estimator, self.cols = estimator, cols

    def fit(self, X, y_):
        self.estimator.fit(X[self.cols].values, y_)
        return self

    def predict(self, X):
        return self.estimator.predict(X[self.cols].values)


class MAEBlend:
    """Finds the weights that give the lowest MAE. The weights are not negative and their sum is 1."""

    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y_):
        preds = X[self.cols].values
        n_models = preds.shape[1]
        result = minimize(lambda w: np.mean(np.abs(preds @ w - y_)), np.ones(n_models) / n_models, method='SLSQP',
                          bounds=[(0, 1)] * n_models, constraints=[{'type': 'eq', 'fun': lambda w: w.sum() - 1}])
        self.weights = result.x
        return self

    def predict(self, X):
        return X[self.cols].values @ self.weights


def make_gbm(max_depth, lr, iters, leaf, l2):
    """Return a gradient-boosted regressor that optimises absolute error."""
    return HistGradientBoostingRegressor(loss='absolute_error', max_depth=max_depth, learning_rate=lr,
                                         max_iter=iters, min_samples_leaf=leaf, l2_regularization=l2,
                                         random_state=SEED)


class ResidualGBM:
    """Takes the mean of the encoder predictions. A small gradient-boosted model then corrects the residual from the fluency features."""

    def __init__(self, base_cols, feat_cols):
        self.base_cols, self.feat_cols = base_cols, feat_cols

    def fit(self, X, y_):
        base = X[self.base_cols].values.mean(axis=1)
        self.gbm = make_gbm(2, 0.03, 120, 25, 2.0).fit(X[self.feat_cols].values, y_ - base)
        return self

    def predict(self, X):
        return X[self.base_cols].values.mean(axis=1) + self.gbm.predict(X[self.feat_cols].values)


def cv_eval(factory, X, y_, n_repeats=3):
    """Score one candidate with repeated 5-fold cross-validation.

    Returns the mean MAE across the repeats and the mean out-of-fold predictions.
    """
    splitter = RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=SEED)
    preds = np.zeros((n_repeats, len(y_)))
    for i, (train_idx, valid_idx) in enumerate(splitter.split(X)):
        model = factory().fit(X.iloc[train_idx], y_[train_idx])
        preds[i // 5, valid_idx] = model.predict(X.iloc[valid_idx])
    fold_maes = [mean_absolute_error(y_, np.clip(p, LABEL_LO, LABEL_HI)) for p in preds]
    return float(np.mean(fold_maes)), np.clip(preds.mean(0), LABEL_LO, LABEL_HI)


pred_cols = list(oof_df.columns)
X_all = pd.concat([oof_df, feat_train], axis=1)
X_test_all = pd.concat([test_df, feat_test], axis=1)

candidates = {f'single:{c}': (lambda c=c: ColumnSubsetModel(RowMeanModel(), [c])) for c in pred_cols}
candidates['mean_of_models'] = lambda: ColumnSubsetModel(RowMeanModel(), pred_cols)
candidates['mae_weighted_blend'] = lambda: MAEBlend(pred_cols)
candidates['ridge_stack'] = lambda: ColumnSubsetModel(Ridge(alpha=1.0), pred_cols)
candidates['gbm_stack_with_features'] = lambda: ColumnSubsetModel(make_gbm(3, 0.05, 150, 20, 1.0), pred_cols + FEATURE_COLS)
candidates['residual_gbm'] = lambda: ResidualGBM(pred_cols, pred_cols + FEATURE_COLS)
candidates['features_only_gbm'] = lambda: ColumnSubsetModel(make_gbm(3, 0.05, 150, 20, 1.0), FEATURE_COLS)

report, cv_oof = [], {}
for name, factory in candidates.items():
    mae, oof_cv = cv_eval(factory, X_all, y)
    cv_oof[name] = oof_cv
    report.append({'candidate': name, 'cv_mae': mae})
report = pd.DataFrame(report).sort_values('cv_mae').reset_index(drop=True)
report.to_csv(RUN_DIR / 'ensemble_report.csv', index=False)
display(report)

ensemble_candidates = [c for c in report['candidate'] if c != 'features_only_gbm']
best_name = ensemble_candidates[0]
print(f'Best candidate: {best_name} (CV MAE {report.set_index("candidate").loc[best_name, "cv_mae"]:.4f})')

,candidate,cv_mae
0,residual_gbm,0.435606
1,gbm_stack_with_features,0.452902
2,single:deberta_v3_large,0.452978
3,mae_weighted_blend,0.453555
4,ridge_stack,0.457591
5,mean_of_models,0.457785
6,single:roberta_large,0.486369
7,features_only_gbm,0.593215


Best candidate: residual_gbm (CV MAE 0.4356)


## 7. Submission file

The selected candidate trains again on all the training rows. The cell limits the test predictions to the label range and moves them to the nearest label on the grid. It writes `submission.csv`, and one submission file for each encoder.


In [8]:
# Move each prediction to the nearest label on the grid, if the labels are a small discrete set and the move lowers the MAE.
grid = np.unique(np.round(y, 4))


def snap(values):
    """Return the nearest grid label for each value."""
    idx = np.abs(values[:, None] - grid[None, :]).argmin(axis=1)
    return grid[idx]


use_snap = False
if len(grid) <= 12:
    raw_mae = mean_absolute_error(y, cv_oof[best_name])
    snap_mae = mean_absolute_error(y, snap(cv_oof[best_name]))
    use_snap = snap_mae < raw_mae - 0.002
    print(f'Label grid {grid.tolist()} | MAE raw={raw_mae:.4f} snapped={snap_mae:.4f} -> snap={use_snap}')
else:
    print(f'{len(grid)} distinct labels: continuous-like, snapping skipped')

final_model = candidates[best_name]().fit(X_all, y)
final_test = np.clip(final_model.predict(X_test_all), LABEL_LO, LABEL_HI)
if use_snap:
    final_test = snap(final_test)

# Give the empty test transcripts the median label of the empty training transcripts.
empty_train = train_rows['transcript'].eq('').values
empty_test = test_rows['transcript'].eq('').values
if KEEP_ZERO_LABELS and empty_train.sum() >= 2 and empty_test.any() and np.median(y[empty_train]) <= 0.5:
    empty_score = float(np.median(y[empty_train]))
    final_test[empty_test] = empty_score
    print(f'Set {int(empty_test.sum())} empty test transcript(s) to {empty_score} (training empties have median label {empty_score})')

submission = pd.DataFrame({'filename': test_rows['filename'].values, 'label': final_test})
submission.to_csv(RUN_DIR / 'submission.csv', index=False)
submission.to_csv('/kaggle/working/submission.csv', index=False)

for name in pred_cols:
    pd.DataFrame({'filename': test_rows['filename'].values, 'label': test_df[name].values}).to_csv(
        RUN_DIR / f'submission_{name}.csv', index=False)

print(f'Saved final submission ({best_name}) with {len(submission)} rows')
print(submission['label'].describe().round(3).to_string())
display(submission.head())

Label grid [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0] | MAE raw=0.4340 snapped=0.4201 -> snap=True
Saved final submission (residual_gbm) with 216 rows
count    216.000
mean       3.255
std        0.841
min        2.000
25%        2.500
50%        3.000
75%        4.000
max        5.000


,filename,label
0,audio_128.wav,3.0
1,audio_156.wav,4.0
2,audio_19.wav,4.0
3,audio_108.wav,2.0
4,audio_206.wav,2.0


## Next steps

1. **Transcript quality.** Make one run with `USE_DISFLUENCY_PROMPT = True` and one run with `False`. Compare the out-of-fold MAE. Also try `WHISPER_MODEL = 'medium'`.
2. **More seeds.** Set `N_SEEDS = 3`. Each fold is cached, so the run continues after an interruption.
3. **Settings.** Try `LOSS_BETA` from 0.1 to 0.5, a different learning rate for each encoder, dropout on, and 8 to 15 epochs.
4. **More encoders.** Add ELECTRA large or DeBERTa-v3 small. A different architecture helps a blend more than one more seed.
5. **Pseudo-labels.** Add the test rows with confident predictions to the training rows. Do this only when the cross-validation setup is stable.
